In [25]:
import numpy as np
import pandas as pd
import NeuralNet
import layer

In [26]:
datasetNames = ['ID number', 'Diagnosis', 
                'Mean Radius', 'Mean Texture', 'Mean Perimeter', 'Mean Area', 'Mean Smoothness', 'Mean Compactness', 'Mean Concavity', 'Mean Concave Points', 'Mean Symmetry', 'Mean Fractal Dimension',
                'SE Radius', 'SE Texture', 'SE Perimeter', 'SE Area', 'SE Smoothness', 'SE Compactness', 'SE Concavity', 'SE Concave Points', 'SE Symmetry', 'SE Fractal Dimension',
                'Worst Radius', 'Worst Texture', 'Worst Perimeter', 'Worst Area', 'Worst Smoothness', 'Worst Compactness', 'Worst Concavity', 'Worst Concave Points', 'Worst Symmetry', 'Worst Fractal Dimension']
dataset = pd.read_csv('data/wdbc.data', names = datasetNames, sep = ',')
dataset = dataset.drop(columns=['ID number'])
dataset.head()

,Diagnosis,Mean Radius,Mean Texture,Mean Perimeter,Mean Area,Mean Smoothness,Mean Compactness,Mean Concavity,Mean Concave Points,Mean Symmetry,...,Worst Radius,Worst Texture,Worst Perimeter,Worst Area,Worst Smoothness,Worst Compactness,Worst Concavity,Worst Concave Points,Worst Symmetry,Worst Fractal Dimension
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [27]:
dataset = dataset.replace({'M':0,'B':1})                                                     
dataset.head()

,Diagnosis,Mean Radius,Mean Texture,Mean Perimeter,Mean Area,Mean Smoothness,Mean Compactness,Mean Concavity,Mean Concave Points,Mean Symmetry,...,Worst Radius,Worst Texture,Worst Perimeter,Worst Area,Worst Smoothness,Worst Compactness,Worst Concavity,Worst Concave Points,Worst Symmetry,Worst Fractal Dimension
0,0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,0,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,0,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,0,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,0,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [28]:
def kfolds(dataset, nfolds):
    folds = []
    foldLength = int(dataset.shape[0]/nfolds)

    dataset = dataset.reindex(np.random.permutation(dataset.index))                                                          
    dataset = dataset.reset_index(drop = True)
    
    idx = 0
    lidx = foldLength - 1
    for i in range(nfolds):
        folds.append(dataset.loc[idx : lidx])
        idx += foldLength
        lidx += foldLength
    return folds

In [29]:
def kfoldsplit(folds):
    train_test = []

    for i in range(len(folds)):
        temp = []
        folds_copy = folds.copy()

        test = folds_copy.pop(i)

        temp.append(pd.concat(folds_copy))
        temp.append(test)
        
        train_test.append(temp)
    return train_test

In [30]:
def kfoldcv(dataset, nfolds = 5):
    folds = kfolds(dataset, nfolds)
    train_test = kfoldsplit(folds)

    accuracies = []
    for i in train_test:
        X_test = np.asarray(i[-1].drop(i[0].columns[0], axis = 1)).T
        y_test = np.array(i[-1].iloc[:,0]).T

        X_mean = np.mean(X_test,axis=1,keepdims=True) #Find the mean of each feature
        X_max = np.max(X_test,axis=1,keepdims=True) #Find the maximum of each feature
        X_test = (X_test-X_mean)/(X_max)

        print("testing data", X_test, y_test)

        X_train = np.array(i[0].drop(i[0].columns[0], axis = 1)).T
        y_train = np.array(i[0].iloc[:,0]).T

        X_mean = np.mean(X_train,axis=1,keepdims=True) #Find the mean of each feature
        X_max = np.max(X_train,axis=1,keepdims=True) #Find the maximum of each feature
        X_normalized = (X_train-X_mean)/(X_max) #Normalizing our dataset by subtracting the mean and dividing by the max

        X_train = X_normalized
        print("training data", X_train, y_train)

        ann = NeuralNet.ANN(0.1, 2, X_train, y_train)
        activations = [2,2]
        ann.setLayers(activations, 2, 1)

        ann.train_sgd()
        accuracy = ann.test(X_test, y_test)
        accuracies.append(accuracy)
    
    return accuracies
kfoldcv(dataset, 4)

testing data [[ 0.00343911  0.08122167  0.09024616 ...  0.05543739  0.01246361
  -0.03652651]
 [-0.07940396  0.06877705 -0.14269884 ...  0.11166377 -0.06313658
  -0.15926199]
 [-0.00284653  0.09616727  0.09025011 ...  0.09090758  0.00458279
  -0.03933569]
 ...
 [-0.14082108  0.42783978  0.12673043 ...  0.56294279 -0.17113011
  -0.08440269]
 [ 0.12809985  0.05106566 -0.01633925 ...  0.0180854  -0.02693145
  -0.05461561]
 [-0.0451852   0.15260839 -0.09002506 ...  0.22876497 -0.04539873
  -0.00589695]] [1 0 1 1 0 1 1 1 0 1 0 1 1 0 1 1 0 0 1 1 0 1 0 0 0 0 1 1 1 1 0 0 0 1 0 0 1
 0 1 1 1 0 1 1 0 0 0 1 1 1 0 1 0 1 0 0 1 1 1 1 1 0 1 1 1 0 1 1 1 1 0 1 1 0
 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 0 1 1 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 0 1 1 0 1 1 0 0 1 1 1 1 0 0 1 0 0 1 1 1 1 1 1 1 1 1 0 1 1]
training data [[ 0.06643226  0.00809003  0.46664571 ...  0.27418751 -0.07942331
  -0.16302345]
 [-0.08668354 -0.13989128  0.17299059 ...  0.06173805 -0.15363874
  -0.03194831]
 [ 0.06937286  0.00337816  0.4

[0.9507042253521126,
 0.9436619718309859,
 0.9436619718309859,
 0.9436619718309859]